# Path B GT - treino e avaliação em sequência (B1, B3, B4)

Este notebook reutiliza o mesmo pré-processamento do Path B e roda, em sequência, o treino e a avaliação dos classificadores B1, B3 e B4 usando crops GT, com células separadas por path.

Passos cobertos:
- Selecionar detector YOLO;
- Pré-processamento dedicado do Path B (idempotente);
- Treinos em GT crops (B1, B3, B4) em células separadas;
- Avaliações combinadas com e sem TTA+WBF em células separadas;
- Ranqueamento final entre B1, B3 e B4.


In [1]:
from pathlib import Path
import os
import sys
import shlex
import subprocess
import shutil
import torch
import pandas as pd
import json

WORKSPACE = Path('/workspace')
REPO_ROOT = WORKSPACE / 'TrashScan'

DATA_DIR = REPO_ROOT / 'data'
TRAIN_DIR = REPO_ROOT / 'train' / 'paths'
EVAL_DIR = REPO_ROOT / 'eval'

EXTERNAL_DIR = WORKSPACE / 'external_datasets'
TACO_DIR = WORKSPACE / 'TACO'
PROCESSED_DIR = WORKSPACE / 'processed_5cls'

DATASET_YAML_PATH_B = PROCESSED_DIR / 'dataset_path_B.yaml'

RUNS_PATH_A_DIR = WORKSPACE / 'runs' / 'path_A'
MLFLOW_DIR = Path('/root/mlflow')
RESULTS_PATH_B_DIR = WORKSPACE / 'results_path_B'

TRAIN_PATH_B_SCRIPT = TRAIN_DIR / 'train_path_B.py'
EVAL_PATH_B_COMBINED_SCRIPT = EVAL_DIR / 'evaluate_path_B_combined.py'

for p in [MLFLOW_DIR, RESULTS_PATH_B_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Python             =', sys.executable)
print('REPO_ROOT          =', REPO_ROOT)
print('PROCESSED_DIR      =', PROCESSED_DIR)
print('DATASET_YAML_PATH_B=', DATASET_YAML_PATH_B)
print('RUNS_PATH_A_DIR    =', RUNS_PATH_A_DIR)
print('RESULTS_PATH_B_DIR =', RESULTS_PATH_B_DIR)
print('TRAIN_PATH_B_SCRIPT=', TRAIN_PATH_B_SCRIPT)
print('EVAL_SCRIPT        =', EVAL_PATH_B_COMBINED_SCRIPT)


def run_cmd(cmd, cwd=WORKSPACE, env=None):
    print('$', ' '.join(shlex.quote(str(x)) for x in cmd))
    return subprocess.run([str(x) for x in cmd], cwd=str(cwd), env=env, check=True)


Python             = /workspace/.venv/bin/python
REPO_ROOT          = /workspace/TrashScan
PROCESSED_DIR      = /workspace/processed_5cls
DATASET_YAML_PATH_B= /workspace/processed_5cls/dataset_path_B.yaml
RUNS_PATH_A_DIR    = /workspace/runs/path_A
RESULTS_PATH_B_DIR = /workspace/results_path_B
TRAIN_PATH_B_SCRIPT= /workspace/TrashScan/train/paths/train_path_B.py
EVAL_SCRIPT        = /workspace/TrashScan/eval/evaluate_path_B_combined.py


In [2]:
if torch.cuda.is_available():
    DEVICE = '0'
    gpu_name = torch.cuda.get_device_properties(0).name
else:
    DEVICE = 'cpu'
    gpu_name = 'cpu'

print('Device:', DEVICE)
print('GPU:', gpu_name)

EPOCHS = 300
BATCH = 8
LR = 5e-5
PATIENCE = 15

EVAL_IMGSZ = 640
EVAL_DET_CONF = 0.001
EVAL_DET_IOU = 0.6

TTA_SCALES = ['512', '640', '768']
TTA_WBF_IOU = '0.55'
TTA_SKIP_BOX_THR = '0.001'

CONFIGS = [
    {
        'key': 'b1',
        'label': 'B1 (resnet50)',
        'classifier': 'resnet50',
        'run_dir': WORKSPACE / 'runs' / 'path_B_gt_b1',
    },
    {
        'key': 'b3',
        'label': 'B3 (vit_b16_imagenet)',
        'classifier': 'vit_b16_imagenet',
        'run_dir': WORKSPACE / 'runs' / 'path_B_gt_b3',
    },
    {
        'key': 'b4',
        'label': 'B4 (vit_l16_imagenet)',
        'classifier': 'vit_l16_imagenet',
        'run_dir': WORKSPACE / 'runs' / 'path_B_gt_b4',
    },
]

B1, B3, B4 = CONFIGS

for cfg in CONFIGS:
    cfg['run_dir'].mkdir(parents=True, exist_ok=True)


Device: 0
GPU: NVIDIA RTX 2000 Ada Generation


## 1) Selecionar detector YOLO

O treino em GT crops não usa o detector para gerar os exemplos. Ainda assim, o script pede `--detector_weights`, e a avaliação combinada precisa do detector.


In [3]:
PREFERRED_DETECTOR = WORKSPACE / 'runs' / 'path_A_5cls' / 'yolov11m_o2o' / 'weights' / 'best.pt'
PATH_A_RUN_DIRS = [
    WORKSPACE / 'runs' / 'path_A_5cls',
    WORKSPACE / 'runs' / 'path_A',
    WORKSPACE / 'runs' / 'path_A_refined_head',
]


def read_yolo_results(run_dir: Path):
    best_pt = run_dir / 'weights' / 'best.pt'
    results_csv = run_dir / 'results.csv'
    metrics_json = run_dir / 'metrics.json'

    if not best_pt.exists():
        return None

    row = {
        'group': run_dir.parent.name,
        'model': run_dir.name,
        'run_dir': run_dir,
        'best_pt': best_pt,
        'mAP50_95': None,
        'mAP50': None,
        'precision': None,
        'recall': None,
        'source': None,
    }

    if results_csv.exists():
        df = pd.read_csv(results_csv)
        df.columns = [c.strip() for c in df.columns]
        map95_col = 'metrics/mAP50-95(B)'
        map50_col = 'metrics/mAP50(B)'
        precision_col = 'metrics/precision(B)'
        recall_col = 'metrics/recall(B)'

        if map95_col in df.columns:
            best_idx = df[map95_col].idxmax()
        elif map50_col in df.columns:
            best_idx = df[map50_col].idxmax()
        else:
            best_idx = df.index[-1]

        best = df.loc[best_idx]
        row['mAP50_95'] = float(best[map95_col]) if map95_col in df.columns else None
        row['mAP50'] = float(best[map50_col]) if map50_col in df.columns else None
        row['precision'] = float(best[precision_col]) if precision_col in df.columns else None
        row['recall'] = float(best[recall_col]) if recall_col in df.columns else None
        row['source'] = 'results.csv'
        return row

    if metrics_json.exists():
        with open(metrics_json, 'r') as f:
            m = json.load(f)
        row['mAP50_95'] = m.get('mAP50_95')
        row['mAP50'] = m.get('mAP50')
        row['precision'] = m.get('precision')
        row['recall'] = m.get('recall')
        row['source'] = 'metrics.json'
        return row

    row['source'] = 'weights_only'
    return row


if PREFERRED_DETECTOR.exists():
    DETECTOR_WEIGHTS = PREFERRED_DETECTOR
    print('Usando detector preferido:', DETECTOR_WEIGHTS)
else:
    records = []
    for base_dir in PATH_A_RUN_DIRS:
        if not base_dir.exists():
            print(f'[warn] Pasta não encontrada: {base_dir}')
            continue
        for run_dir in sorted(base_dir.iterdir()):
            if run_dir.is_dir():
                rec = read_yolo_results(run_dir)
                if rec is not None:
                    records.append(rec)

    df_detectors = pd.DataFrame(records)
    if df_detectors.empty:
        raise FileNotFoundError(
            'Nenhum detector com weights/best.pt foi encontrado em: '
            + ', '.join(str(p) for p in PATH_A_RUN_DIRS)
        )

    df_ranked = df_detectors.copy()
    df_ranked['rank_score'] = df_ranked['mAP50_95'].fillna(df_ranked['mAP50']).fillna(-1)
    df_ranked = df_ranked.sort_values(
        by=['rank_score', 'mAP50', 'precision', 'recall'],
        ascending=False,
        na_position='last',
    ).reset_index(drop=True)

    display(df_ranked[['group', 'model', 'mAP50_95', 'mAP50', 'precision', 'recall', 'source', 'best_pt']])
    DETECTOR_WEIGHTS = Path(df_ranked.iloc[0]['best_pt'])
    print('Melhor detector encontrado:', DETECTOR_WEIGHTS)

if not DETECTOR_WEIGHTS.exists():
    raise FileNotFoundError(f'Detector não encontrado: {DETECTOR_WEIGHTS}')


Usando detector preferido: /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt


## 2) Pré-processamento dedicado do Path B

Esta etapa prepara os dados do Path B a partir do merged_data em `/workspace/processed_5cls/merged_data`,
gerando `images`, `labels`, `crops` e `dataset_path_B.yaml` dentro do root `/workspace/processed_5cls`.

A célula abaixo é idempotente: se a estrutura `train/val/test/path_B/{images,labels,crops}` e o YAML já existirem, ela só reutiliza o dataset. Para recriar o pré-processamento manualmente, altere `FORCE_PREPROCESS_PATH_B` para `True`.


In [4]:
# Pré-processamento do Path B a partir do merged_data. Rode uma vez; os outros treinos GT reutilizam o mesmo root.
FORCE_PREPROCESS_PATH_B = False

PREPROCESS_PATH_B_SCRIPT = DATA_DIR / 'preprocess.py'
MERGED_ROOT = PROCESSED_DIR / 'merged_data'
REQUIRED_PATH_B_ITEMS = [DATASET_YAML_PATH_B]

for split in ['train', 'val', 'test']:
    for subdir in ['images', 'labels', 'crops']:
        REQUIRED_PATH_B_ITEMS.append(PROCESSED_DIR / split / 'path_B' / subdir)

missing_path_b_items = [p for p in REQUIRED_PATH_B_ITEMS if not p.exists()]

if FORCE_PREPROCESS_PATH_B or missing_path_b_items:
    if FORCE_PREPROCESS_PATH_B:
        print('FORCE_PREPROCESS_PATH_B=True; executando pré-processamento do Path B.')
    else:
        print('Pré-processamento do Path B ausente ou incompleto. Itens faltantes:')
        for p in missing_path_b_items:
            print(' -', p)

    if not MERGED_ROOT.exists():
        raise FileNotFoundError(f'merged_data não encontrado: {MERGED_ROOT}')
    if not PREPROCESS_PATH_B_SCRIPT.exists():
        raise FileNotFoundError(f'Script de pré-processamento não encontrado: {PREPROCESS_PATH_B_SCRIPT}')

    run_cmd([
        sys.executable, str(PREPROCESS_PATH_B_SCRIPT),
        '--taco_root', str(MERGED_ROOT),
        '--output_root', str(PROCESSED_DIR),
        '--path', 'B',
    ])
else:
    print('Pré-processamento do Path B já encontrado:', PROCESSED_DIR)


Pré-processamento do Path B já encontrado: /workspace/processed_5cls


## 3) Conferir crops GT

Este notebook treina em `CropDataset`, então ele depende de `train/val/test/path_B/crops/{class_idx}`.


In [5]:
for split in ['train', 'val', 'test']:
    crop_root = PROCESSED_DIR / split / 'path_B' / 'crops'
    if not crop_root.exists():
        raise FileNotFoundError(f'Crops GT não encontrados: {crop_root}')

    counts = {}
    for cls_dir in sorted(crop_root.iterdir()):
        if cls_dir.is_dir():
            counts[cls_dir.name] = len(list(cls_dir.glob('*.jpg')))

    print(split, crop_root)
    print('  total:', sum(counts.values()), 'por classe:', counts)


train /workspace/processed_5cls/train/path_B/crops
  total: 15678 por classe: {'0': 6025, '1': 1307, '2': 1245, '3': 500, '4': 6601}
val /workspace/processed_5cls/val/path_B/crops
  total: 3447 por classe: {'0': 1411, '1': 267, '2': 282, '3': 107, '4': 1380}
test /workspace/processed_5cls/test/path_B/crops
  total: 3208 por classe: {'0': 1312, '1': 280, '2': 265, '3': 88, '4': 1263}


## 4) Treinos em GT crops (B1, B3, B4)

Sem `--use_yolo_crops`. O classificador é treinado nos crops GT pré-gerados.
Parâmetros fixos: epochs=300, batch=8, patience=15.


In [ ]:
print('---', B1['label'], '---')
print('Parâmetros de treino:')
print('Detector weights:', DETECTOR_WEIGHTS)
print('Crops dir:', PROCESSED_DIR)
print('Output dir:', B1['run_dir'])
print('Classifier:', B1['classifier'])
print('Epochs:', EPOCHS)
print('Batch:', BATCH)
print('Learning rate:', LR)
print('Patience:', PATIENCE)
print('Device:', DEVICE)
print('Use YOLO crops:', False)

run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--crops_dir', str(PROCESSED_DIR),
    '--output', str(B1['run_dir']),
    '--classifiers', B1['classifier'],
    '--epochs', str(EPOCHS),
    '--batch', str(BATCH),
    '--lr', str(LR),
    '--patience', str(PATIENCE),
    '--device', str(DEVICE),
])


--- B1 (resnet50) ---
Parâmetros de treino:
Detector weights: /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt
Crops dir: /workspace/processed_5cls
Output dir: /workspace/runs/path_B_gt_b1
Classifier: resnet50
Epochs: 300
Batch: 8
Learning rate: 5e-05
Patience: 15
Device: 0
Use YOLO crops: False
TTA+WBF no treino: False
$ /workspace/.venv/bin/python /workspace/TrashScan/train/paths/train_path_B.py --detector_weights /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt --crops_dir /workspace/processed_5cls --output /workspace/runs/path_B_gt_b1 --classifiers resnet50 --epochs 300 --batch 8 --lr 5e-05 --patience 15 --device 0
Device : cuda:0
GPU    : NVIDIA RTX 2000 Ada Generation  VRAM: 16.8GB
Class weights: {'plastic': 0.511, 'paper': 1.098, 'metal': 1.125, 'glass': 1.777, 'other': 0.489}

  Mode: loading pre-generated GT crops from disk
  [train] 15678 GT crops across 5 classes
  [val] 3447 GT crops across 5 classes
  [test] 3208 GT crops across 5 classes

─────────────────

  Ep 1/300 train:  48%|████▊     | 932/1960 [00:51<00:51, 20.09it/s]

In [ ]:
print('---', B3['label'], '---')
print('Parâmetros de treino:')
print('Detector weights:', DETECTOR_WEIGHTS)
print('Crops dir:', PROCESSED_DIR)
print('Output dir:', B3['run_dir'])
print('Classifier:', B3['classifier'])
print('Epochs:', EPOCHS)
print('Batch:', BATCH)
print('Learning rate:', LR)
print('Patience:', PATIENCE)
print('Device:', DEVICE)
print('Use YOLO crops:', False)

run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--crops_dir', str(PROCESSED_DIR),
    '--output', str(B3['run_dir']),
    '--classifiers', B3['classifier'],
    '--epochs', str(EPOCHS),
    '--batch', str(BATCH),
    '--lr', str(LR),
    '--patience', str(PATIENCE),
    '--device', str(DEVICE),
])


In [ ]:
print('---', B4['label'], '---')
print('Parâmetros de treino:')
print('Detector weights:', DETECTOR_WEIGHTS)
print('Crops dir:', PROCESSED_DIR)
print('Output dir:', B4['run_dir'])
print('Classifier:', B4['classifier'])
print('Epochs:', EPOCHS)
print('Batch:', BATCH)
print('Learning rate:', LR)
print('Patience:', PATIENCE)
print('Device:', DEVICE)
print('Use YOLO crops:', False)
print('TTA+WBF no treino:', False)

run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--crops_dir', str(PROCESSED_DIR),
    '--output', str(B4['run_dir']),
    '--classifiers', B4['classifier'],
    '--epochs', str(EPOCHS),
    '--batch', str(BATCH),
    '--lr', str(LR),
    '--patience', str(PATIENCE),
    '--device', str(DEVICE),
])


## 5) Resumo do treino (B1, B3, B4)


In [ ]:
run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--crops_dir', str(PROCESSED_DIR),
    '--output', str(B1['run_dir']),
    '--summarize',
])


In [ ]:
run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--crops_dir', str(PROCESSED_DIR),
    '--output', str(B3['run_dir']),
    '--summarize',
])


In [ ]:
run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--crops_dir', str(PROCESSED_DIR),
    '--output', str(B4['run_dir']),
    '--summarize',
])


## 6) Avaliação combinada sem TTA+WBF (B1, B3, B4)

Esta seção avalia detector + classificador no modo standard.


In [ ]:
CLASSIFIER_DIR = B1['run_dir']
OUTPUT_DIR_STANDARD = RESULTS_PATH_B_DIR / 'b1_5cls_gt_standard'

cmd = [
    sys.executable, str(EVAL_PATH_B_COMBINED_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--classifier_dir', str(CLASSIFIER_DIR),
    '--classifiers', B1['classifier'],
    '--data_yaml', str(DATASET_YAML_PATH_B),
    '--output', str(OUTPUT_DIR_STANDARD),
    '--device', str(DEVICE),
    '--imgsz', str(EVAL_IMGSZ),
    '--det_conf', str(EVAL_DET_CONF),
    '--det_iou', str(EVAL_DET_IOU),
]

run_cmd(cmd)


In [ ]:
CLASSIFIER_DIR = B3['run_dir']
OUTPUT_DIR_STANDARD = RESULTS_PATH_B_DIR / 'b3_5cls_gt_standard'

cmd = [
    sys.executable, str(EVAL_PATH_B_COMBINED_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--classifier_dir', str(CLASSIFIER_DIR),
    '--classifiers', B3['classifier'],
    '--data_yaml', str(DATASET_YAML_PATH_B),
    '--output', str(OUTPUT_DIR_STANDARD),
    '--device', str(DEVICE),
    '--imgsz', str(EVAL_IMGSZ),
    '--det_conf', str(EVAL_DET_CONF),
    '--det_iou', str(EVAL_DET_IOU),
]

run_cmd(cmd)


In [ ]:
CLASSIFIER_DIR = B4['run_dir']
OUTPUT_DIR_STANDARD = RESULTS_PATH_B_DIR / 'b4_5cls_gt_standard'

cmd = [
    sys.executable, str(EVAL_PATH_B_COMBINED_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--classifier_dir', str(CLASSIFIER_DIR),
    '--classifiers', B4['classifier'],
    '--data_yaml', str(DATASET_YAML_PATH_B),
    '--output', str(OUTPUT_DIR_STANDARD),
    '--device', str(DEVICE),
    '--imgsz', str(EVAL_IMGSZ),
    '--det_conf', str(EVAL_DET_CONF),
    '--det_iou', str(EVAL_DET_IOU),
]

run_cmd(cmd)


## 7) Avaliação combinada com TTA+WBF (B1, B3, B4)

Esta seção avalia o mesmo classificador treinado em GT crops, mas com o detector usando TTA multi-scale + WBF.


In [ ]:
CLASSIFIER_DIR = B1['run_dir']
OUTPUT_DIR_TTA_WBF = RESULTS_PATH_B_DIR / 'b1_5cls_gt_tta_wbf'

cmd = [
    sys.executable, str(EVAL_PATH_B_COMBINED_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--classifier_dir', str(CLASSIFIER_DIR),
    '--classifiers', B1['classifier'],
    '--data_yaml', str(DATASET_YAML_PATH_B),
    '--output', str(OUTPUT_DIR_TTA_WBF),
    '--device', str(DEVICE),
    '--imgsz', str(EVAL_IMGSZ),
    '--det_conf', str(EVAL_DET_CONF),
    '--det_iou', str(EVAL_DET_IOU),
    '--use_tta_wbf',
    '--tta_scales', *TTA_SCALES,
    '--tta_flip',
    '--tta_wbf_iou', TTA_WBF_IOU,
    '--tta_skip_box_thr', TTA_SKIP_BOX_THR,
]

run_cmd(cmd)


In [ ]:
CLASSIFIER_DIR = B3['run_dir']
OUTPUT_DIR_TTA_WBF = RESULTS_PATH_B_DIR / 'b3_5cls_gt_tta_wbf'

cmd = [
    sys.executable, str(EVAL_PATH_B_COMBINED_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--classifier_dir', str(CLASSIFIER_DIR),
    '--classifiers', B3['classifier'],
    '--data_yaml', str(DATASET_YAML_PATH_B),
    '--output', str(OUTPUT_DIR_TTA_WBF),
    '--device', str(DEVICE),
    '--imgsz', str(EVAL_IMGSZ),
    '--det_conf', str(EVAL_DET_CONF),
    '--det_iou', str(EVAL_DET_IOU),
    '--use_tta_wbf',
    '--tta_scales', *TTA_SCALES,
    '--tta_flip',
    '--tta_wbf_iou', TTA_WBF_IOU,
    '--tta_skip_box_thr', TTA_SKIP_BOX_THR,
]

run_cmd(cmd)


In [ ]:
CLASSIFIER_DIR = B4['run_dir']
OUTPUT_DIR_TTA_WBF = RESULTS_PATH_B_DIR / 'b4_5cls_gt_tta_wbf'

cmd = [
    sys.executable, str(EVAL_PATH_B_COMBINED_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--classifier_dir', str(CLASSIFIER_DIR),
    '--classifiers', B4['classifier'],
    '--data_yaml', str(DATASET_YAML_PATH_B),
    '--output', str(OUTPUT_DIR_TTA_WBF),
    '--device', str(DEVICE),
    '--imgsz', str(EVAL_IMGSZ),
    '--det_conf', str(EVAL_DET_CONF),
    '--det_iou', str(EVAL_DET_IOU),
    '--use_tta_wbf',
    '--tta_scales', *TTA_SCALES,
    '--tta_flip',
    '--tta_wbf_iou', TTA_WBF_IOU,
    '--tta_skip_box_thr', TTA_SKIP_BOX_THR,
]

run_cmd(cmd)


## 8) Ranqueamento entre B1, B3 e B4


In [ ]:
rows = []

def add_summary_rows(summary_path: Path, model_label: str, eval_mode: str):
    if not summary_path.exists():
        print(f'[warn] Resultado não encontrado para {model_label} ({eval_mode}): {summary_path}')
        return
    data = json.loads(summary_path.read_text())
    if isinstance(data, list):
        for row in data:
            row = dict(row)
            row['model'] = model_label
            row['eval_mode'] = eval_mode
            rows.append(row)
    else:
        row = dict(data)
        row['model'] = model_label
        row['eval_mode'] = eval_mode
        rows.append(row)

for cfg in CONFIGS:
    add_summary_rows(
        RESULTS_PATH_B_DIR / (cfg['key'] + '_5cls_gt_standard') / 'path_B_combined_summary.json',
        cfg['label'],
        'standard',
    )
    add_summary_rows(
        RESULTS_PATH_B_DIR / (cfg['key'] + '_5cls_gt_tta_wbf') / 'path_B_combined_summary.json',
        cfg['label'],
        'tta_wbf',
    )

if not rows:
    print('Nenhum resumo encontrado ainda.')
else:
    df = pd.DataFrame(rows)
    score_col = 'mAP50'
    cols = [c for c in ['model', 'eval_mode', 'classifier', score_col, 'mAP50', 'mAP50_95', 'fps', 'inference_mode'] if c in df.columns]

    if score_col not in df.columns:
        print('[warn] Coluna mAP50 não encontrada; não é possível ranquear.')
    else:
        for mode in ['standard', 'tta_wbf']:
            df_mode = df[df['eval_mode'] == mode].copy()
            if df_mode.empty:
                print(f'[warn] Sem dados para modo: {mode}')
                continue
            df_mode = df_mode.sort_values(by=[score_col], ascending=False).reset_index(drop=True)
            print(f'Ranking ({mode}) - ordenado por {score_col}')
            display(df_mode[cols])
